# 第6章 Notebook：ドア付近滞留者の行動ルール

対応章: [`../chapters/06_agent_boarding_behavior.md`](../chapters/06_agent_boarding_behavior.md)

この notebook は、卒業研究準備セミナーの数値実験用である。上から順に実行すれば、本文で説明した図を再現できる。設定パラメータは上部のセルにまとめてある。乱数は seed を固定している。

## 1. ライブラリ読み込み

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

## 2. 第5章のモデルを再利用（行動を「譲る確率」で表す）

行動タイプを `stander_yield`（ドア付近の人が道を譲る確率）で簡略表現する。固定型=0.0、横移動型=0.6、一時降車型=0.95。値が大きいほど降車流を妨げない。

In [ ]:
def make_grid(H=11, W=21, door_col=10, door_half=1):
    passable = np.ones((H, W), bool)
    cy = H // 2
    wall = np.ones(H, bool)
    wall[cy - door_half: cy + door_half + 1] = False
    passable[wall, door_col] = False   # wall except door rows
    return passable, door_col, cy

def place_agents(passable, door_col, n_alight, n_board, seed):
    rng = np.random.default_rng(seed)
    H, W = passable.shape
    occ = np.zeros((H, W), bool)
    train = [(r, c) for r in range(H) for c in range(door_col) if passable[r, c]]
    plat = [(r, c) for r in range(H) for c in range(door_col + 1, W) if passable[r, c]]
    rng.shuffle(train); rng.shuffle(plat)
    agents = []
    for (r, c) in train[:n_alight]:
        agents.append({'pos': (r, c), 'type': 'alight', 'done': False}); occ[r, c] = True
    for (r, c) in plat[:n_board]:
        agents.append({'pos': (r, c), 'type': 'board', 'done': False}); occ[r, c] = True
    return agents, occ

def simulate(H=11, W=21, door_col=10, door_half=1, n_alight=14, n_board=14,
             standers=0, stander_yield=0.0, max_steps=600, seed=0):
    rng = np.random.default_rng(seed)
    passable, dc, cy = make_grid(H, W, door_col, door_half)
    agents, occ = place_agents(passable, dc, n_alight, n_board, seed)
    # fixed standers just outside the door; keep the centre door row open
    cand = [(cy + 1, dc + 1), (cy - 1, dc + 1), (cy, dc + 2),
            (cy + 1, dc + 2), (cy - 1, dc + 2)]
    stander_set = set()
    for (r, c) in cand:
        if len(stander_set) >= standers:
            break
        if 0 <= r < H and 0 <= c < W and passable[r, c] and not occ[r, c]:
            stander_set.add((r, c))

    def blocked(nr, nc):
        if occ[nr, nc]:
            return True
        if (nr, nc) in stander_set:
            return rng.random() > stander_yield   # fixed (yield=0) always blocks
        return False

    density = np.zeros((H, W))
    steps = 0
    while steps < max_steps and any(not a['done'] for a in agents):
        order = [a for a in agents if not a['done']]
        rng.shuffle(order)
        for a in order:
            r, c = a['pos']
            tc = W - 1 if a['type'] == 'alight' else 0
            cur = (abs(c - tc), abs(r - cy))
            best = None
            for dr, dcc in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                nr, nc = r + dr, c + dcc
                if 0 <= nr < H and 0 <= nc < W and passable[nr, nc] and not blocked(nr, nc):
                    sc = (abs(nc - tc), abs(nr - cy))
                    if sc < cur and (best is None or sc < best[0]):
                        best = (sc, nr, nc)
            if best is not None and best[1:] not in stander_set:
                nr, nc = best[1], best[2]
                occ[r, c] = False; occ[nr, nc] = True; a['pos'] = (nr, nc)
                r, c = nr, nc
            if a['type'] == 'alight' and c == W - 1:
                a['done'] = True; occ[r, c] = False
            elif a['type'] == 'board' and c == 0:
                a['done'] = True; occ[r, c] = False
        density += occ
        steps += 1
    return steps, density, passable

## 3-4. 行動タイプ別に複数回シミュレーション

In [ ]:
behaviors = {'fixed': 0.0, 'sidestep': 0.6, 'temp_alight': 0.95}
N_RUNS = 20
results = {}
for name, y in behaviors.items():
    dwell = [simulate(standers=2, stander_yield=y, seed=s)[0] for s in range(N_RUNS)]
    results[name] = dwell
    print(f'{name:12s} mean dwell = {np.mean(dwell):.1f} +/- {np.std(dwell):.1f}')

## 5-6. 停車時間の分布（箱ひげ図）

In [ ]:
plt.figure(figsize=(6.5, 4))
plt.boxplot([results[n] for n in behaviors], labels=list(behaviors))
plt.ylabel('dwell steps'); plt.title('dwell time distribution by behavior')
plt.tight_layout(); plt.show()

## 7. 密度ヒートマップ（行動タイプ別）

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for a, (name, y) in zip(ax, behaviors.items()):
    _, dens, _ = simulate(standers=2, stander_yield=y, seed=0)
    im = a.imshow(dens, cmap='viridis'); a.set_title(f'{name}')
    a.axvline(10, color='w', ls='--', lw=1)
plt.tight_layout(); plt.show()

## 8. 卒研で実測動画に接続するためのメモ

実際の乗降動画から人の軌跡を抽出し、停車時間の分布をこのモデルと比較できる。効用関数の重みを実データに合わせて推定すれば、行動の逆問題になる。

## 9. 課題（自分で変更する）

1. `behaviors` に中間的な譲り確率（例 0.3）を追加し、分布を比較せよ。
2. `N_RUNS` を増やすと平均・分散の推定はどう安定するか確かめよ。

In [ ]:
# === 課題セル ===
extra = [simulate(standers=2, stander_yield=0.3, seed=s)[0] for s in range(N_RUNS)]
print('yield=0.3 mean dwell =', round(np.mean(extra), 1))